# Adaptive Immune Profiling Challenge 2025 Exploration


In [ ]:
# !pip install torch
# !pip install transformers bitsandbytes accelerate
# !pip install datasets
# !export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# !export TOKENIZERS_PARALLELISM=True

import pandas as pd
from pathlib import Path

INPUT_PATH = Path("/kaggle/input/adaptive-immune-profiling-challenge-2025")
OUTPUT_PATH = Path("/kaggle/working")
TRAIN_DIR_PATH = INPUT_PATH / "train_datasets" / "train_datasets"
TEST_DIR_PATH = INPUT_PATH / "test_datasets" / "test_datasets"

assert INPUT_PATH.exists() and OUTPUT_PATH.exists()
assert TRAIN_DIR_PATH.exists() and TRAIN_DIR_PATH.is_dir()
assert TEST_DIR_PATH.exists() and TEST_DIR_PATH.is_dir()

DATA_COLS = ["junction_aa", "v_call", "j_call"]
DATA_COLS_EXTRA = ["d_call", "templates"]

META_COLS = ["repertoire_id", "filename", "label_positive"]
META_COLS_EXTRA = [
    "study_group_description",
    "sex",
    "age",
    "A",
    "B",
    "C",
    "DPA1",
    "DPB1",
    "DQA1",
    "DQB1",
    "DRB1",
    "DRB3",
    "DRB4",
    "DRB5",
    "race",
    "sequencing_run_id",
]

NEW_DATA_COLS = ["dir", "repertoire_id"]
NEW_META_COLS = ["dir"]
NEW_COLS = list(set(NEW_DATA_COLS + NEW_META_COLS))

# **Utils Helper Functions**


In [ ]:
def print_tree(path: Path, prefix="", max_files=5):
    contents = list(path.iterdir())

    # Separate directories and files
    dirs = [c for c in contents if c.is_dir()]
    files = [c for c in contents if not c.is_dir()]

    # Always show all directories first
    for idx, child in enumerate(dirs):
        connector = "└── " if idx == len(dirs) - 1 and not files else "├── "
        print(prefix + connector + child.name)
        extension = "    " if idx == len(dirs) - 1 and not files else "│   "
        print_tree(child, prefix + extension, max_files=max_files)

    # Show only first `max_files` files
    for idx, child in enumerate(files[:max_files]):
        connector = "└── " if idx == len(files[:max_files]) - 1 else "├── "
        print(prefix + connector + child.name)

    # If there are more files than max_files, indicate how many are hidden
    if len(files) > max_files:
        print(prefix + f"└── ... ({len(files) - max_files} more files)")


def check_metadata(data_path: Path = TRAIN_DIR_PATH):
    # Checks unique metadata
    for dir in data_path.iterdir():
        if dir.is_dir():
            meta_file_path = dir / "metadata.csv"
            meta = pd.read_table(meta_file_path, sep=",")
            total_data = sum(
                [1 for f in dir.iterdir() if f.is_file() and f.stem != "metadata"]
            )
            print(
                f">>>>>>>>>>>>>>>>>> Metadata Dir {dir.stem}\n> META COLS: {list(meta.columns)}\n> DATA COUNT: {total_data}"
            )

In [ ]:
# Overview of Data Directory Path
print_tree(INPUT_PATH)

In [ ]:
# Check existence of metadata per training dataset. Dataset `train_dataset_8` is the dataset with full complete meta data features
check_metadata()

# Data Loader


In [ ]:
# DATA LOADER given input list of train dataset directories
# NOTE: Data batches is grouped by the directory name "train_dataset_*" and the data is stored as CSV files

META_FILENAME = "metadata"  # metadata filename per train_dataset_* directory
MAX_TRAIN_REPER = (
    10  # total to repertoires_id batches (.tsv files) to load from a train directory
)
SELECT_TRAIN_DIR = [
    "train_dataset_8",
    "train_dataset_7",
    "train_dataset_3",
]  # list of train dirs to load

DEMO_REPER_ID = "7dd34f5f47de76229ab8bf8300fd84a3"
DEMO_DIR = "train_dataset_8"
NA_VALS = ["unknown"]


class FeatEncoderConfig:
    label_positive: dict = {False: 0, True: 1}
    sex: dict = {"M": 0, "F": 1}
    study_group_description: dict = {"FDR": 0, "T1D": 1, "CTRL": 2}


class LoadRepertoireDataset:
    def __init__(self, train_dir_path: Path = TRAIN_DIR_PATH, **kwargs):
        self.load_dataset(
            train_dir=train_dir_path,
            select_train_dir=kwargs.get("select_train_dir", SELECT_TRAIN_DIR),
        )

        print(f"Train dataset loaded: {self.__dict__.keys()}")

    def load_dataset(self, train_dir: Path, select_train_dir: list):
        # Loads data and merges with the metadata

        meta_ = []
        for dir in train_dir.iterdir():
            if dir.is_dir() and dir.stem in select_train_dir:
                # Loads metadata
                meta_file_path = dir / "metadata.csv"
                if not meta_file_path.exists():
                    raise FileExistsError(
                        f"Metadata {meta_file_path} does not exist in attempted dir."
                    )
                else:
                    meta = pd.read_table(meta_file_path, sep=",")
                    meta["dir"] = dir.stem
                    meta_.append(meta)

                # Loads datasaets
                samples = []
                for idx, f in enumerate(dir.iterdir()):
                    if f.stem != META_FILENAME:
                        df = pd.read_table(f)
                        df["repertoire_id"] = f.stem
                        df["dir"] = dir.stem

                        meta_info = meta[
                            (meta["repertoire_id"] == f.stem)
                            & (meta["dir"] == dir.stem)
                        ]
                        if meta_info["label_positive"].shape[0] > 1:
                            raise RuntimeError(
                                f"Expected 1 row exist for unique pair of (repertoire_id, dir) but filtered data has more than 1 row. The filtered data: \n{meta_info}"
                            )

                        df["label_positive"] = meta_info["label_positive"].iloc[0]

                        if common := list(
                            set(META_COLS_EXTRA) & set(meta_info.columns)
                        ):
                            for col in common:
                                df[col] = meta_info[col].iloc[0]

                        samples.append(df)

                    if idx >= MAX_TRAIN_REPER:
                        break

                if samples:
                    dfs = pd.concat(samples)
                    dfs.reset_index(drop=True, inplace=True)
                    setattr(self, dir.stem, dfs)
                    print(
                        f"Loaded training dir {dir.stem} into dataclass successfully! \nDATA: {list(getattr(self, dir.stem).columns)} {getattr(self, dir.stem).shape[0]}"
                    )
                    self._post_process_load(dir.stem)
                else:
                    raise RuntimeError("Error loading training dataset")

        if meta_:
            meta_df = pd.concat(meta_)
            setattr(self, META_FILENAME, meta_df)
        else:
            raise FileExistsError(
                "Error loading training metadata files. Could not concatenate empty metadata."
            )

    def _post_process_load(self, train_attr_name: str):
        # Post-processor func after loading and merging datataset

        data: pd.DataFrame = getattr(self, train_attr_name)

        # Converts the dataset to expected / appropiate val
        if "d_call" in data.columns:
            data["d_call"].replace("unknown", pd.NA, inplace=True)
        else:
            data["d_call"] = pd.NA

        if data["label_positive"].dtype != bool:
            data["label_positive"] = data["label_positive"].astype(bool)

        # Add new features
        data["len_junction_aa"] = [len(list(i)) for i in data["junction_aa"]]
        data["beta_count"] = data.groupby(["v_call", "j_call"])["v_call"].transform(
            "count"
        )
        data["beta_proba"] = data["beta_count"] / data["v_call"].nunique()

        # v_to_j = data.groupby(["v_call", "j_call"])["v_call"].size().reset_index(name="v_j")
        # data = data.merge(v_to_j, on=["v_call", "j_call"], how="left")

        # Label Encoding features
        data["label_positive"] = data["label_positive"].map(
            FeatEncoderConfig.label_positive
        )
        if "sex" in data.columns:
            data["sex"] = data["sex"].map(FeatEncoderConfig.sex)
        if "study_group_description" in data.columns:
            data["study_group_description"].map(
                FeatEncoderConfig.study_group_description
            )

        setattr(self, train_attr_name, data)


ds = LoadRepertoireDataset()

## Quick look


In [ ]:
ds.train_dataset_8["junction_aa"].value_counts()  # pyright: ignore[reportUndefinedVariable]

In [ ]:
(
    ds.train_dataset_8["j_call"].value_counts(),
    ds.train_dataset_8["v_call"].value_counts(),
    ds.train_dataset_8["d_call"].value_counts(),
)

In [ ]:
ds.train_dataset_8["age"].value_counts(), ds.train_dataset_8["sex"].value_counts()

In [ ]:
(
    ds.train_dataset_8["A"].value_counts(),
    ds.train_dataset_8["B"].value_counts(),
    ds.train_dataset_8["C"].value_counts(),
)

In [ ]:
ds.train_dataset_8.columns

In [ ]:
(
    ds.train_dataset_8["junction_aa"].nunique(),
    ds.train_dataset_8["v_call"].nunique(),
    ds.train_dataset_8["j_call"].nunique(),
    ds.train_dataset_8["d_call"].nunique(),
)

In [ ]:
ds.train_dataset_8.loc[ds.train_dataset_8["len_junction_aa"] == 1]

# **Epilogue**


In [ ]:
import os

# Enable expandable segments to reduce fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

### Data Processor

In [ ]:
# Data batches is grouped by the directory name "train_dataset_*" and the data is stored as CSV files

import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, BertModel

META_FILENAME = "metadata"  # metadata filename per train_dataset_* directory
MAX_TRAIN_REPER = (
    3  # total to repertoires_id batches (.tsv files) to load from a train directory
)
SELECT_TRAIN_DIR = [
    "train_dataset_8",
    "train_dataset_7",
    "train_dataset_3",
]  # list of train dirs to load
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
TCR_BERT_MODEL_ID = "wukevin/tcr-bert"
TCR_BERT_TOKENIZER_KWARG = {
    "padding": True,
    "truncation": True,
    "return_tensors": "pt",
    "max_length": 20,
}
INFERENCE_BATCH_SIZE = 100000
TO_TENSOR = True


class FeatEncoderConfig:
    label_positive: dict = {False: 0, True: 1}
    sex: dict = {"M": 0, "F": 1}
    study_group_description: dict = {"FDR": 0, "T1D": 1, "CTRL": 2}


class RepertoireDataset(Dataset):
    def __init__(self, path: Path = TRAIN_DIR_PATH, **kwargs):
        self.path = path
        self.dirs = kwargs.get("select_dir", SELECT_TRAIN_DIR)
        self.max_train_samples = kwargs.get("max_train_samples", MAX_TRAIN_REPER)
        self.meta_file_name = kwargs.get("meta_filename", META_FILENAME)
        self.device = kwargs.get("device", DEVICE)
        self.embedding_batch_size = kwargs.get(
            "embedding_batch_size", INFERENCE_BATCH_SIZE
        )  # For GPU batching
        self._to_tensor = kwargs.get("to_tensor", TO_TENSOR)
        self._data = {}

        if self._to_tensor is True:
            self.tokenizer_kwargs = kwargs.get(
                "tokenizer_kwargs", TCR_BERT_TOKENIZER_KWARG
            )
            print(f"Loading TCR-BERT model on device: {self.device}")
            self._tokenizer = AutoTokenizer.from_pretrained(
                TCR_BERT_MODEL_ID, do_lower_case=True
            )
            self._model = BertModel.from_pretrained(TCR_BERT_MODEL_ID)
            self._model.to(self.device)
            self._model.eval()
            print(f"✓ Model loaded on {self.device}")

    def _fetch_embeddings(self, junction_aa: pd.Series):
        """Generate embeddings in batches to avoid OOM on GPU."""
        N = junction_aa.shape[0]
        M = self.tokenizer_kwargs.get("max_length")

        seq = junction_aa.map(lambda x: " ".join(str(x))).values.tolist()
        all_embeddings = []

        # Process in batches
        for i in tqdm(
            range(0, N, self.embedding_batch_size), desc="Processing junction_aa tcr"
        ):
            batch_seq = seq[i : i + self.embedding_batch_size]

            # Tokenize
            inputs = self._tokenizer(batch_seq, **self.tokenizer_kwargs)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            assert inputs["input_ids"].shape[-1] == M, (
                f"Expected input_ids shape to be (_, {M}) but returned tokenized input_ids: {inputs['input_ids'].shape}"
            )

            with torch.no_grad():
                outputs = self._model(**inputs)
                batch_embeddings = outputs.pooler_output.cpu()
                all_embeddings.append(batch_embeddings)

            # Clear GPU cache
            del inputs, outputs
            if self.device.type == "cuda":
                torch.cuda.empty_cache()

            tqdm.write(
                f"Processed {min(i + self.embedding_batch_size, N)}/{N} sequences"
            )

        embeddings = torch.cat(all_embeddings, dim=0)
        assert embeddings.shape[0] == N and embeddings.shape[-1] == 768, (
            f"Expected {N} embeddings, got {embeddings.shape[0]}"
        )
        return embeddings

    def __len__(self):
        return sum([i.shape[0] for _, i in self._data.items()])

    def _load_dataset(self, train_dir_path: Path, meta_file_path: Path):
        meta = pd.read_table(meta_file_path, sep=",")
        meta["dir"] = train_dir_path.stem

        samples = []
        for idx, f in enumerate(train_dir_path.iterdir()):
            if f.stem != self.meta_file_name:
                df = pd.read_table(f)
                df["repertoire_id"] = f.stem
                df["dir"] = train_dir_path.stem
                samples.append(df)

            if idx >= self.max_train_samples:
                print(
                    f"Finished loading {self.max_train_samples} of train dir {train_dir_path.stem}"
                )
                break

        if samples:
            dfs = pd.concat(samples)
            dfs.reset_index(drop=True, inplace=True)
            merged_df = dfs.merge(
                meta, on=["repertoire_id", "dir"], how="outer", indicator=True
            )
            merged = merged_df[merged_df["_merge"] == "both"]
            merged = merged.drop(columns=["_merge"])
            if merged.shape[0] != dfs.shape[0] or merged.shape[-1] != abs(
                (dfs.shape[-1] + meta.shape[-1] - 2)
            ):
                raise RuntimeError(
                    f"Merge Error. Expected Merged shape to be ({dfs.shape[0], abs((dfs.shape[-1] + meta.shape[-1] - 2))}). But merged dataframe returned shape {merged.shape}"
                )

            self._data[train_dir_path.stem] = merged
            self._transform_features(train_dir_path)

    def _transform_features(self, train_dir_path: Path):
        if train_dir_path.stem not in self._data:
            raise RuntimeError

        data = self._data[train_dir_path.stem]
        if "d_call" in data.columns:
            data["d_call"] = data["d_call"].replace("unknown", pd.NA)
        else:
            data["d_call"] = pd.NA

        if data["label_positive"].dtype != bool:
            data["label_positive"] = data["label_positive"].astype(bool)

        # Add new features
        data["len_junction_aa"] = [len(list(i)) for i in data["junction_aa"]]
        data["beta_count"] = data.groupby(["v_call", "j_call"])["v_call"].transform(
            "count"
        )
        data["beta_proba"] = data["beta_count"] / data["v_call"].nunique()
        # Label Encoding features
        data["label_positive"] = data["label_positive"].map(
            FeatEncoderConfig.label_positive
        )
        if "sex" in data.columns:
            data["sex"] = data["sex"].map(FeatEncoderConfig.sex)
        if "study_group_description" in data.columns:
            data["study_group_description"].map(
                FeatEncoderConfig.study_group_description
            )

        self._data[train_dir_path.stem] = data

    def _preprocess_dataset(self, train_dir_path: Path):
        # TODO: create processing func for vectorizing (v_call, j_call) beta tcr sequencies
        # TODO: return (v_call, j_call) embeddings
        # TODO: return meta features
        if train_dir_path.stem not in self._data:
            raise RuntimeError
        if not hasattr(self, "_model") or not hasattr(self, "_tokenizer"):
            raise AttributeError

        data = self._data[train_dir_path.stem]

        inputs = self._fetch_embeddings(junction_aa=data["junction_aa"])
        labels = torch.tensor(
            data["label_positive"].values, dtype=torch.float32, requires_grad=False
        )

        return {"inputs": inputs, "labels": labels}

    def __getitem__(self, idx: int):
        if idx < 0 or idx >= len(self.dirs):
            return None

        path = self.path / self.dirs[idx]
        if not path.exists():
            raise FileExistsError(f"Error finding data path: {path}")

        meta_file_path = path / f"{self.meta_file_name}.csv"
        if not meta_file_path.exists():
            raise FileExistsError(
                f"Metadata {meta_file_path} does not exist in attempted dir."
            )

        if path.stem not in self._data:
            self._load_dataset(train_dir_path=path, meta_file_path=meta_file_path)

        print(f"Cached data size {len(self)}")
        if self._to_tensor is True:
            return self._preprocess_dataset(train_dir_path=path)

        return self._data[path.stem]


ds = RepertoireDataset()

In [ ]:
df = ds[0]
junction_aa = df["junction_aa"].map(lambda x: " ".join(str(x)))

Finished loading 3 of train dir train_dataset_8
Cached data size 322263
Cached data size 322263


In [ ]:
df

,junction_aa,v_call,j_call,d_call,templates,repertoire_id,dir,filename,label_positive,study_group_description,...,DPB1,DQA1,DQB1,DRB1,DRB3,DRB4,DRB5,len_junction_aa,beta_count,beta_proba
123,CAAADGNSYEQYF,TCRBV06-05,TCRBJ02-07,TCRBD02-01,1.0,247445210b805700759d0e7fbd4d7c16,train_dataset_8,247445210b805700759d0e7fbd4d7c16.tsv,0,FDR,...,401;1301,401;301,302;402,801;402,9901;9901,103;9901,9901;9901,13,2729,38.985714
124,CAAAGGENTEAFF,TCRBV05-04,TCRBJ01-01,<NA>,1.0,247445210b805700759d0e7fbd4d7c16,train_dataset_8,247445210b805700759d0e7fbd4d7c16.tsv,0,FDR,...,401;1301,401;301,302;402,801;402,9901;9901,103;9901,9901;9901,13,897,12.814286
125,CAAAGTGRGFF,TCRBV28-01,TCRBJ02-01,TCRBD01-01,1.0,247445210b805700759d0e7fbd4d7c16,train_dataset_8,247445210b805700759d0e7fbd4d7c16.tsv,0,FDR,...,401;1301,401;301,302;402,801;402,9901;9901,103;9901,9901;9901,11,1020,14.571429
126,CAAALGLANTGELFF,TCRBV10-02,TCRBJ02-02,TCRBD02-01,1.0,247445210b805700759d0e7fbd4d7c16,train_dataset_8,247445210b805700759d0e7fbd4d7c16.tsv,0,FDR,...,401;1301,401;301,302;402,801;402,9901;9901,103;9901,9901;9901,15,104,1.485714
127,CAAAQITGASTEAFF,TCRBV11,TCRBJ01-01,<NA>,1.0,247445210b805700759d0e7fbd4d7c16,train_dataset_8,247445210b805700759d0e7fbd4d7c16.tsv,0,FDR,...,401;1301,401;301,302;402,801;402,9901;9901,103;9901,9901;9901,15,814,11.628571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
322878,YATYSGEYNEQFF,TCRBV10-03,TCRBJ02-01,TCRBD02-01,1.0,b034a69e3b0e6622994461266c06a7b7,train_dataset_8,b034a69e3b0e6622994461266c06a7b7.tsv,0,FDR,...,401;201,301;501,302;201,301;401,101;9901,103;9901,9901;9901,13,737,10.528571
322879,YAWSLKGYGYTF,TCRBV30-01,TCRBJ01-02,<NA>,1.0,b034a69e3b0e6622994461266c06a7b7,train_dataset_8,b034a69e3b0e6622994461266c06a7b7.tsv,0,FDR,...,401;201,301;501,302;201,301;401,101;9901,103;9901,9901;9901,12,1184,16.914286
322880,YAWSVAGLVVSNQPQHF,TCRBV30-01,TCRBJ01-05,TCRBD01-01,1.0,b034a69e3b0e6622994461266c06a7b7,train_dataset_8,b034a69e3b0e6622994461266c06a7b7.tsv,0,FDR,...,401;201,301;501,302;201,301;401,101;9901,103;9901,9901;9901,17,831,11.871429
322881,YHGGNEQFF,TCRBV14-01,TCRBJ02-01,TCRBD02-01,1.0,b034a69e3b0e6622994461266c06a7b7,train_dataset_8,b034a69e3b0e6622994461266c06a7b7.tsv,0,FDR,...,401;201,301;501,302;201,301;401,101;9901,103;9901,9901;9901,9,512,7.314286


In [28]:
# Load pre-trained model TCRBert to generate base feature vector
TCR_BERT_MODEL_ID = "wukevin/tcr-bert"
tokenizer = AutoTokenizer.from_pretrained(TCR_BERT_MODEL_ID)
model = BertModel.from_pretrained(TCR_BERT_MODEL_ID)
model = model.to(DEVICE)
model.eval()  # set to evaluation mode if you don't plan to fine‑tune

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(26, 768, padding_idx=21)
    (position_embeddings): Embedding(64, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
 

**Monitoring Vec Embeddings vs GPU**

The `get_embeddings_batched` monitors how embeddings / data batches (or caches) are processed on GPU.

In [ ]:
# IMPORTANT: Process in batches to avoid OOM on Tesla T4 (14GB)
INFERENCE_BATCH_SIZE = 10000  # Adjust based on your dataset size


def get_embeddings_batched(
    sequences, tokenizer, model, device, batch_size=INFERENCE_BATCH_SIZE, max_length=20
):
    """
    Generate BERT embeddings in batches with proper device management.

    Args:
        sequences: List of amino acid sequences
        tokenizer: Pre-loaded tokenizer
        model: Pre-loaded BERT model (already on device)
        device: torch.device object
        batch_size: Number of sequences per batch
        max_length: Max sequence length for tokenization

    Returns:
        torch.Tensor: Embeddings on CPU (N, hidden_dim)
    """
    model.eval()
    all_embeddings = []

    for i in range(0, len(sequences), batch_size):
        batch_seqs = sequences[i : i + batch_size]

        # Tokenize and move to GPU
        inputs = tokenizer(
            batch_seqs,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate embeddings
        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.pooler_output.cpu()  # Move to CPU immediately
            all_embeddings.append(batch_embeddings)

        # Clear GPU memory
        del inputs, outputs
        if device.type == "cuda":
            torch.cuda.empty_cache()

        if (i // batch_size) % 10 == 0:
            print(
                f"Processed {min(i + batch_size, len(sequences))}/{len(sequences)} sequences"
            )

    return torch.cat(all_embeddings, dim=0)


# Generate embeddings for your dataset
embeddings = get_embeddings_batched(
    sequences=junction_aa.values.tolist(),
    tokenizer=tokenizer,
    model=model,
    device=DEVICE,
)

print(f"✓ Embeddings shape: {embeddings.shape}")
print(f"✓ Device used: {DEVICE}")
print(f"✓ Embeddings dtype: {embeddings.dtype}")

Processed 10000/1029478 sequences
Processed 110000/1029478 sequences
Processed 110000/1029478 sequences
Processed 210000/1029478 sequences
Processed 210000/1029478 sequences
Processed 310000/1029478 sequences
Processed 310000/1029478 sequences
Processed 410000/1029478 sequences
Processed 410000/1029478 sequences
Processed 510000/1029478 sequences
Processed 510000/1029478 sequences
Processed 610000/1029478 sequences
Processed 610000/1029478 sequences
Processed 710000/1029478 sequences
Processed 710000/1029478 sequences
Processed 810000/1029478 sequences
Processed 810000/1029478 sequences
Processed 910000/1029478 sequences
Processed 910000/1029478 sequences
Processed 1010000/1029478 sequences
Processed 1010000/1029478 sequences
✓ Embeddings shape: torch.Size([1029478, 768])
✓ Device used: cuda
✓ Embeddings dtype: torch.float32
✓ Embeddings shape: torch.Size([1029478, 768])
✓ Device used: cuda
✓ Embeddings dtype: torch.float32


In [59]:
embeddings.shape

torch.Size([1029478, 768])

In [ ]:
# Monitor GPU memory usage
print("=" * 60)
print("GPU MEMORY USAGE")
print("=" * 60)
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.3f} GB")
    print(f"Memory Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.3f} GB")
    print(
        f"Max Memory Allocated: {torch.cuda.max_memory_allocated(0) / 1024**3:.3f} GB"
    )

    # Clear cache if needed
    # torch.cuda.empty_cache()
    # print("✓ Cache cleared")
else:
    print("Device: CPU")

print("=" * 60)

CUDA available: True
Device: Tesla T4
Memory Allocated: 0.4283027648925781 GB
Memory Reserved: 1.681640625 GB


In [ ]:
# Data Processing functions

BATCH_SIZE = 1
HIDDEN_DIM = 768


def run_pretrained_tcr(tokenizer, model, junction_aa):
    # junction_aa processor

    max_length = len(list(junction_aa))
    seq = " ".join(junction_aa)
    inputs = tokenizer(
        seq, padding=True, truncation=True, max_length=max_length, return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)
        # outputs.last_hidden_state: shape (batch_size, seq_length, hidden_size)
        # outputs.pooler_output: shape (batch_size, hidden_size), the [CLS] embedding
        print(outputs.keys())
        embeddings = (
            outputs.pooler_output.cpu().numpy()
        )  # [1, 768] = [BATCH_SIZE, HIDDEN_DIM]

    return embeddings


def run_pretrained_tcr_series(tokenizer, model, junction_aa: pd.Series):
    seq = junction_aa.map(lambda x: " ".join(str(x)))
    inputs = tokenizer(inputs, **TCR_BERT_TOKENIZER_KWARG)
    pass


# output = run_pretrained_tcr(tokenizer,model,junction_aa="ALCQQLEVGEGQPQHF")

# Scratchpad for dummy runs


In [ ]:
OUTPUT_DIM = 2
INPUT_DIM = 768
HIDDEN_DIM = INPUT_DIM + 384
V_CALLS = 100


class RepertoireMLP(nn.Module):
    def __init__(
        self,
        input_dim: int = INPUT_DIM,
        hidden_dim: int = HIDDEN_DIM,
        output_dim: int = OUTPUT_DIM,
    ):
        super(RepertoireMLP, self).__init__()

        self.f1 = nn.Linear(input_dim, hidden_dim)
        self.f2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, inputs):
        z0 = self.relu(self.f1(inputs))
        print(z0.shape)
        z1 = self.f2(z0)
        print(z1.shape)
        return z1.sigmoid(-1)


In [ ]:
import torch
import torch.nn as nn

# Configuration
embed_dim = 128
num_heads = 1
seq_length = 20
batch_size = 1

# MultiheadAttention layer with batch_first=True
mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)

# Input: (batch_size, seq_len, embed_dim)
x = torch.rand(batch_size, seq_length, embed_dim)

# Causal mask: prevents attending to future positions
# Mask shape: (seq_len, seq_len)
mask = torch.triu(torch.ones(seq_length, seq_length) * float("-inf"), diagonal=1)

# Forward pass: self-attention
attn_output, attn_weights = mha(x, x, x, attn_mask=mask)

print("Attention output shape:", attn_output.shape)  # (batch_size, seq_len, embed_dim)
print(
    "Attention weights shape:", attn_weights.shape
)  # (batch_size, num_heads, seq_len, seq_len)

In [ ]:
import torch
import torch.nn as nn

# Example V and J gene calls (categorical)
v_calls = ["TRBV5-1", "TRBV7-2", "TRBV12-3"]  # example sequence length = 3
j_calls = ["TRBJ1-1", "TRBJ2-3", "TRBJ2-7"]

# Create a vocabulary for V and J genes
v_vocab = {gene: i for i, gene in enumerate(set(v_calls))}
j_vocab = {gene: i for i, gene in enumerate(set(j_calls))}

# Convert calls to indices
v_indices = torch.tensor([v_vocab[v] for v in v_calls])
j_indices = torch.tensor([j_vocab[j] for j in j_calls])

# Embedding layers
v_embed = nn.Embedding(len(v_vocab), embed_dim)  # 16-dim embedding
j_embed = nn.Embedding(len(j_vocab), embed_dim)

# Get embeddings
v_vecs = v_embed(v_indices)  # (seq_len, embed_dim)
j_vecs = j_embed(j_indices)

# Stack V and J embeddings into a sequence
# seq = torch.stack([v_vecs, j_vecs], dim=0)  # (2, seq_len, embed_dim)
# or concatenate along sequence dimension
seq = torch.cat([v_vecs, j_vecs], dim=0)  # (seq_len*2, embed_dim)
seq = seq.unsqueeze(0)  # (batch_size, seq_length*2, embed_dim)

# Attention layer
mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)

# Self-attention
attn_output, attn_weights = mha(seq, seq, seq)

print("attn_output shape:", attn_output.shape)  # (seq_len*2, 1, embed_dim)
print("attn_weights shape:", attn_weights.shape)

In [ ]:
torch.where(attn_weights > 0.2, torch.tensor(1.0), torch.tensor(0.0))

In [ ]:
attn_weights.max(), attn_weights.mean(), attn_weights.min()

In [326]:
torch.where(attn_weights > 0.2, torch.tensor(1.0), torch.tensor(0.0)).sum()

In [ ]:
# Refactored comp monitor to add to local workspace

import torch
import functools


def profile_memory(per_batch=False):
    """
    Decorator to profile GPU/CPU memory usage before, after, and optionally per batch.
    Tracks peak GPU memory usage across all batches.

    Args:
        per_batch (bool): If True, prints memory usage after every batch.
    """

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            peak_memory = 0.0  # Track peak memory

            def print_memory_usage(label=""):
                nonlocal peak_memory
                if torch.cuda.is_available():
                    allocated = torch.cuda.memory_allocated(0) / 1024**3
                    reserved = torch.cuda.memory_reserved(0) / 1024**3
                    max_alloc = torch.cuda.max_memory_allocated(0) / 1024**3
                    peak_memory = max(peak_memory, allocated, max_alloc)
                    print("=" * 60)
                    print(f"GPU MEMORY USAGE {label}")
                    print("=" * 60)
                    print(f"CUDA available: True")
                    print(f"Device: {torch.cuda.get_device_name(0)}")
                    print(f"Memory Allocated: {allocated:.3f} GB")
                    print(f"Memory Reserved: {reserved:.3f} GB")
                    print(f"Max Memory Allocated: {max_alloc:.3f} GB")
                else:
                    print("=" * 60)
                    print(f"CPU MEMORY USAGE {label}")
                    print("=" * 60)
                    print("Device: CPU")
                print("=" * 60)

            # Inject memory callback for per-batch monitoring
            if per_batch:
                kwargs["_memory_callback"] = print_memory_usage

            print("Before function execution:")
            print_memory_usage("BEFORE")

            result = func(*args, **kwargs)

            print("After function execution:")
            print_memory_usage("AFTER")

            print("=" * 60)
            print(f"PEAK GPU MEMORY DURING EXECUTION: {peak_memory:.3f} GB")
            print("=" * 60)

            return result

        return wrapper

    return decorator


@profile_memory(per_batch=True)
def get_embeddings_batched(
    sequences,
    tokenizer,
    model,
    device,
    batch_size=INFERENCE_BATCH_SIZE,
    max_length=20,
    _memory_callback=None,
):
    model.eval()
    all_embeddings = []

    for i in range(0, len(sequences), batch_size):
        batch_seqs = sequences[i : i + batch_size]

        inputs = tokenizer(
            batch_seqs,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.pooler_output.cpu()
            all_embeddings.append(batch_embeddings)

        del inputs, outputs
        if device.type == "cuda":
            torch.cuda.empty_cache()

        if (i // batch_size) % 10 == 0:
            print(
                f"Processed {min(i + batch_size, len(sequences))}/{len(sequences)} sequences"
            )

        if _memory_callback:
            _memory_callback(f"AFTER BATCH {i // batch_size + 1}")

    return torch.cat(all_embeddings, dim=0)